# Reading QIF

Every notebook in this workshop starts the same way: open a QIF file with
Python's own `ElementTree` and follow the references inside it. This notebook follows the references one lookup at a
time. Later notebooks use helper functions that do the same lookups for the whole
document.

The helpers live in [`qifkit.py`](qifkit.py), next to this notebook. It is
written to be read easily. Where this notebook uses a helper, it
prints the helper's source.

No 3D here. Setup is in the [README](../README.md).

## Setup


`%autoreload 2` re-reads [`qifkit.py`](qifkit.py) before every cell, so an edit
to it takes effect without restarting the kernel.

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

import qifkit

pd.set_option('display.width', 140)

import copy
import inspect
import xml.etree.ElementTree as ET

from IPython.display import Code, display

from qifkit import NS, text

# Print elements as the file writes them, rather than with an ns0: prefix on
# every tag.
ET.register_namespace('', NS['qif'])


def source(function):
    # Show a qifkit function's source, highlighted.
    display(Code(inspect.getsource(function), language='python'))


def xml(element):
    # Show an element as XML, indented from the left margin whatever the file's
    # own layout. Indent a copy, so the parsed document is left alone.
    shown = copy.deepcopy(element)
    ET.indent(shown)
    display(Code(ET.tostring(shown, encoding='unicode').strip(), language='xml'))

## The document

`000454_D-MV-FH-2` is the demo file for the whole workshop. It is a complete QIF
document: the part's B-rep geometry, the PMI (the tolerances on the drawing)
**and** the CMM's results for 15 manufactured parts, all in one file.

Everything in it that other elements need to refer to carries an `id`. `load`
parses the file and indexes every one of those elements by id, once. Every "follow this
reference" below is then a dictionary lookup.

In [2]:
MODEL = '000454_D-MV-FH-2'
doc = qifkit.load(MODEL)

print(f'{MODEL}: {len(doc.by_id)} identified elements')
source(qifkit.load)

000454_D-MV-FH-2: 4626 identified elements


def load(model):
    # Parse a model from models/ and index it by id.
    #
    # QIF is a web of references: a measurement names its characteristic by id,
    # the characteristic names its feature by id, and so on down to the face.
    # Following any of those links means finding an element by its id. Searching
    # the whole tree each time would rescan a multi-megabyte document for every
    # hop, so index every element that carries an id once, up front. After that
    # each hop is a dictionary lookup.
    path = MODELS / f'{model}.qif'
    root = ET.parse(path).getroot()
    by_id = {el.get('id'): el for el in root.iter() if el.get('id') is not None}
    return Doc(model=model, path=path, root=root, by_id=by_id)

The top level of the document, and how many children each section has (`QPId` holds a single value, so it shows that instead).
**Features** and **Characteristics** are the design: what the part is and what the
drawing asks of it. **Results** is what the CMM found.

In [3]:
for section in doc.root:
    print(f'{section.tag.split("}")[1]:<22}{len(section) or section.text:>4}')

QPId                  52863fb4-ed44-4d80-9cf5-0844fed367c6
Attributes              12
VersionHistory           2
Version                  1
Header                   3
StandardsDefinitions     1
FileUnits                2
DatumDefinitions         3
DatumReferenceFrames     3
Product                  9
Features                 3
Characteristics          4
Results                  2
UserDataXML              1


## Following a tolerance to the part

Take one callout from the drawing: **Ø8.73 ±0.254, applied to four holes**. In QIF
a callout is a `CharacteristicNominal`. Its tolerance is in a separate
`CharacteristicDefinition`, and the features it applies to are `FeatureNominal`s,
each of which names the faces of the B-rep it covers.

```mermaid
flowchart LR
    N["CharacteristicNominal<br/>the callout"] -->|CharacteristicDefinitionId| D["CharacteristicDefinition<br/>the tolerance"]
    N -->|FeatureNominalIds| F["FeatureNominal<br/>every feature it covers"]
    F -->|EntityInternalIds| G["face id = qif_id"]
```

Every arrow is an id. Find the callout by its name first:

In [4]:
callout = next(nominal for nominal in doc.root.findall('.//qif:CharacteristicNominals/*', NS)
               if text(nominal, './qif:Name') == 'Radial Dimension (188)')
xml(callout)

<DiameterCharacteristicNominal xmlns="http://qifstandards.org/xsd/qif3" xmlns:ns1="urn:x-3dv:userdata" id="3527">
  <Attributes n="7">
    <AttributeStr name="_3dv.PersistentIdNx" value="RM000454_D.prt R002c550e00000100" />
    <AttributeI1 name="_3dv.FeatureCounter" value="4" />
    <AttributeUser name="_3dv.PmiReportableAdvanced" nameUserAttribute="AttributePmiReportable">
      <UserDataXML>
        <ns1:ReportableStates n="2">
          <ns1:Reportable reportType="56" val="1" />
          <ns1:Reportable reportType="4" val="1" />
        </ns1:ReportableStates>
      </UserDataXML>
    </AttributeUser>
    <AttributeStr name="_3dv.FeatureDetailCode" value="DIA" />
    <AttributeStr name="CAMC" value="No" />
    <AttributeStr name="_3dv.TextPrefix" value="4X" />
    <AttributeI1 name="DimensionDisplayMode" value="1" />
  </Attributes>
  <CharacteristicDefinitionId>3526</CharacteristicDefinitionId>
  <FeatureNominalIds n="4">
    <Id>5547</Id>
    <Id>5548</Id>
    <Id>5550</Id>
    <Id>5549</Id>
  </FeatureNominalIds>
  <Name>Radial Dimension (188)</Name>
  <CharacteristicDesignator>
    <Designator>13</Designator>
    <UUID>60746213-fab1-4979-bebe-62aed832d17a</UUID>
  </CharacteristicDesignator>
  <TargetValue decimalPlaces="2">8.731249999999999</TargetValue>
</DiameterCharacteristicNominal>

Most of these are exporter attributes. The elements used below are
`CharacteristicDefinitionId`, `FeatureNominalIds` and `TargetValue`.

**Hop 1: the tolerance.** `CharacteristicDefinitionId` is an id; look it up.

In [5]:
definition = doc.by_id[text(callout, './qif:CharacteristicDefinitionId')]
xml(definition)
print('tolerance (low, high):', qifkit.tolerance(definition))

<DiameterCharacteristicDefinition xmlns="http://qifstandards.org/xsd/qif3" id="3526">
  <CharacteristicDesignator>
    <Designator />
    <UUID>c105c7b8-91e4-43b3-84cf-87e927b96f9d</UUID>
  </CharacteristicDesignator>
  <Tolerance>
    <MaxValue decimalPlaces="2">0.254</MaxValue>
    <MinValue decimalPlaces="2">-0.254</MinValue>
    <DefinedAsLimit>false</DefinedAsLimit>
  </Tolerance>
</DiameterCharacteristicDefinition>

tolerance (low, high): (-0.254, 0.254)


A diameter carries `MaxValue` and `MinValue` as offsets either side of the nominal.
A geometric tolerance such as flatness carries a single `ToleranceValue` instead.
`tolerance` reads both shapes:

In [6]:
source(qifkit.tolerance)

def tolerance(definition):
    # (low, high) offsets from nominal, or None where nothing is toleranced.
    #
    # Two shapes. A dimensional characteristic (a diameter, a width) carries
    # Tolerance/MaxValue and MinValue as signed offsets either side of the
    # nominal. A geometric one (flatness, position) carries a single
    # ToleranceValue, which is a zone: its measured value is already a
    # deviation, so the band runs from zero.
    if definition is None or definition.find('./qif:NonTolerance', NS) is not None:
        return None

    single = text(definition, './qif:ToleranceValue')
    if single is not None:
        return (0.0, float(single))

    high = text(definition, './qif:Tolerance/qif:MaxValue')
    low = text(definition, './qif:Tolerance/qif:MinValue')
    if high is None or low is None:
        return None
    if float(high) - float(low) <= 0:
        # A basic dimension written as Max == Min == 0 by an exporter that did
        # not emit NonTolerance. That is a zero band, not an infinitely tight one.
        return None
    return (float(low), float(high))

**Hop 2: the features, and their faces.** `FeatureNominalIds` lists four ids, one
per hole. Each `FeatureNominal` lists the B-rep faces it covers in
`EntityInternalIds`.

In [7]:
for feature_id in callout.findall('./qif:FeatureNominalIds/qif:Id', NS):
    feature = doc.by_id[feature_id.text]
    faces = [i.text for i in feature.findall('./qif:EntityInternalIds/qif:Id', NS)]
    print(f'{feature.tag.split("}")[1]} {feature_id.text} -> faces {faces}')

xml(doc.by_id[callout.find('./qif:FeatureNominalIds/qif:Id', NS).text])

CylinderFeatureNominal 5547 -> faces ['1116']
CylinderFeatureNominal 5548 -> faces ['1110']
CylinderFeatureNominal 5550 -> faces ['1122']
CylinderFeatureNominal 5549 -> faces ['1128']


<CylinderFeatureNominal xmlns="http://qifstandards.org/xsd/qif3" id="5547">
  <Attributes n="1">
    <AttributeStr name="_3dv.TrueGeomAlgo" value="Y14.5-2009" />
  </Attributes>
  <Name>Nominal 5547</Name>
  <UUID>59b49af0-73e4-409d-bec5-f25888aa6a67</UUID>
  <FeatureDefinitionId>5546</FeatureDefinitionId>
  <EntityInternalIds n="1">
    <Id>1116</Id>
  </EntityInternalIds>
  <SubstituteFeatureAlgorithm>
    <SubstituteFeatureAlgorithmEnum>LEASTSQUARES</SubstituteFeatureAlgorithmEnum>
  </SubstituteFeatureAlgorithm>
  <Axis>
    <AxisPoint>-38 38.000000000000099 0</AxisPoint>
    <Direction>0 0 -1</Direction>
  </Axis>
</CylinderFeatureNominal>

Those face ids are the end of the chain. The mesher stamps the same number on every
triangle it makes from a face, as `qif_id`. A face id read out of the XML is therefore
directly a set of triangles on the 3D part. That is the join notebook 01 draws.

Features can nest: a hole pattern is a feature whose members are features. So the
helper that collects face ids has to recurse:

In [8]:
source(qifkit.entity_ids)
[qifkit.entity_ids(doc, i.text) for i in callout.findall('./qif:FeatureNominalIds/qif:Id', NS)]

def entity_ids(doc, feature_nominal_id, seen=None):
    # Geometry entity (face) ids under a feature nominal, recursing into groups.
    #
    # Features nest: a pattern of holes is a feature whose members are themselves
    # features, and only the innermost ones name faces. So this has to recurse.
    # `seen` guards against a document that points a group back at itself.
    #
    # A dangling reference (an id nothing in the document carries) yields no
    # faces rather than an error. Real exports have them.
    seen = set() if seen is None else seen
    if feature_nominal_id in seen:
        return []
    seen.add(feature_nominal_id)

    feature_nominal = doc.by_id.get(feature_nominal_id)
    if feature_nominal is None:
        return []

    ids = [i.text for i in feature_nominal.findall('./qif:EntityInternalIds/qif:Id', NS)]
    for member in feature_nominal.findall('./qif:FeatureNominalIds/qif:Id', NS):
        ids.extend(entity_ids(doc, member.text, seen))
    return ids

[['1116'], ['1110'], ['1122'], ['1128']]

## Following a measurement to the part

A measurement is **one application** of a callout, on one part. It does not point
at the callout directly: it points at a `CharacteristicItem`, and the item points
at both the callout and the one feature it was applied to.

```mermaid
flowchart LR
    M["CharacteristicMeasurement<br/>value, status"] -->|CharacteristicItemId| I["CharacteristicItem<br/>one application"]
    I -->|CharacteristicNominalId| N["CharacteristicNominal<br/>the callout"]
    I -->|FeatureItemIds| FI["FeatureItem"]
    FI -->|FeatureNominalId| F["FeatureNominal<br/>this application's feature"]
    F -->|EntityInternalIds| G["face id = qif_id"]
```

The difference matters. Reached through the callout, a measurement of one hole maps to
all four holes. Reached through the item, it maps only to its own hole.

Take the first measurement of this callout in the first set of results:

In [9]:
results = doc.root.find('.//qif:MeasurementResults', NS)

measurement = next(
    m for m in results.findall('.//qif:CharacteristicMeasurements/*', NS)
    if text(doc.by_id.get(text(m, './qif:CharacteristicItemId')),
            './qif:CharacteristicNominalId') == callout.get('id'))
xml(measurement)

<DiameterCharacteristicMeasurement xmlns="http://qifstandards.org/xsd/qif3" id="6210">
  <Status>
    <CharacteristicStatusEnum>PASS</CharacteristicStatusEnum>
  </Status>
  <CharacteristicItemId>5677</CharacteristicItemId>
  <FeatureMeasurementIds n="1">
    <Id>6177</Id>
  </FeatureMeasurementIds>
  <ActualComponentId>9054</ActualComponentId>
  <Value decimalPlaces="3">8.800173554977009</Value>
  <MaxValue decimalPlaces="3">8.829921837406101</MaxValue>
  <MinValue decimalPlaces="3">8.829921837406101</MinValue>
</DiameterCharacteristicMeasurement>

**Serial number.** `ActualComponentId` names an `ActualComponent`, which carries the
serial number.

In [10]:
component = doc.by_id[text(measurement, './qif:ActualComponentId')]
xml(component)

<ActualComponent xmlns="http://qifstandards.org/xsd/qif3" id="9054">
  <SerialNumber>SN#1</SerialNumber>
  <Status>
    <InspectionStatusEnum>UNDEFINED</InspectionStatusEnum>
  </Status>
</ActualComponent>

Not every file records it there. The other sample part leaves it off the
measurement and puts it on the `MeasurementResults` instead. Both are valid QIF, so
`serial_number` tries both:

In [11]:
source(qifkit.serial_number)
qifkit.serial_number(doc, results, measurement)

def serial_number(doc, measurement_results, measurement):
    # Serial of the part a measurement belongs to.
    #
    # The two sample parts record this two valid ways. 000454_D hangs
    # ActualComponentIds off each MeasurementResults; 000473_A omits that and
    # puts an ActualComponentId on every individual measurement instead. Try
    # both.
    for component_id in (text(measurement_results, './qif:ActualComponentIds/qif:Id'),
                         text(measurement, './qif:ActualComponentId')):
        if component_id is not None:
            serial = text(doc.by_id.get(component_id), './qif:SerialNumber')
            if serial:
                return serial
    return None

'SN#1'

**Hops 1–4: item, feature item, feature nominal, faces.**

In [12]:
item = doc.by_id[text(measurement, './qif:CharacteristicItemId')]
xml(item)

feature_item = doc.by_id[text(item, './qif:FeatureItemIds/qif:Id')]
xml(feature_item)

feature_nominal_id = text(feature_item, './qif:FeatureNominalId')
print('value:', text(measurement, './qif:Value'),
      '| callout:', text(doc.by_id[text(item, './qif:CharacteristicNominalId')], './qif:Name'),
      '| faces:', qifkit.entity_ids(doc, feature_nominal_id))

<DiameterCharacteristicItem xmlns="http://qifstandards.org/xsd/qif3" id="5677">
  <Name>Diameter Item 5677</Name>
  <CharacteristicDesignator>
    <Designator>13.1</Designator>
    <UUID>06330ff0-79e0-4e74-b7a6-6cb09c137c03</UUID>
  </CharacteristicDesignator>
  <FeatureItemIds n="1">
    <Id>5678</Id>
  </FeatureItemIds>
  <CharacteristicNominalId>3527</CharacteristicNominalId>
</DiameterCharacteristicItem>

<CylinderFeatureItem xmlns="http://qifstandards.org/xsd/qif3" id="5678">
  <FeatureNominalId>5547</FeatureNominalId>
  <FeatureName>CylinderItem5678</FeatureName>
  <DeterminationMode>
    <Checked />
  </DeterminationMode>
</CylinderFeatureItem>

value: 8.800173554977009 | callout: Radial Dimension (188) | faces: ['1116']


One hole, one face. The callout covered four.

## The same walk, for every measurement

`measurement_table` repeats those hops for every measurement in the file and returns
one row per measurement. Notebooks 02 onward start from this table. Each step in it
appears in the walk above:

In [13]:
source(qifkit.measurement_table)

def measurement_table(doc):
    # One row per CharacteristicMeasurement: one application of a callout, on
    # one part. Nothing is filtered out here; see set_aside for that.
    #
    # Columns:
    #   serial          which part (SN#1...)
    #   characteristic  the CharacteristicItem's name: one application
    #   name            the CharacteristicNominal's name: the callout
    #   type            Diameter, Position, Flatness...
    #   nominal         target value; None for geometric tolerances
    #   low, high       tolerance offsets from nominal (see tolerance)
    #   value           what the CMM reported
    #   status_from_file  the status the file records: PASS, FAIL or UNDEFINED
    #   entities        face ids this application covers, as a sorted tuple
    #   fraction        see fraction_of_band
    rows = []
    for results in doc.root.findall('.//qif:MeasurementResults', NS):
        for measurement in results.findall('.//qif:CharacteristicMeasurements/*', NS):
            item = doc.by_id.get(text(measurement, './qif:CharacteristicItemId'))
            nominal = doc.by_id.get(text(item, './qif:CharacteristicNominalId'))
            if nominal is None:
                continue
            definition = doc.by_id.get(text(nominal, './qif:CharacteristicDefinitionId'))

            value = text(measurement, './qif:Value')
            target = text(nominal, './qif:TargetValue')
            band = tolerance(definition)

            # Geometry through the item, not the nominal: the nominal lists every
            # feature the callout covers, the item only the one it was applied to.
            entities = []
            for feature_item_id in item.findall('./qif:FeatureItemIds/qif:Id', NS):
                feature_item = doc.by_id.get(feature_item_id.text)
                entities.extend(entity_ids(doc, text(feature_item, './qif:FeatureNominalId')))

            rows.append({
                'serial': serial_number(doc, results, measurement),
                'characteristic': text(item, './qif:Name'),
                'name': text(nominal, './qif:Name'),
                'type': measurement.tag.split('}')[1].replace('CharacteristicMeasurement', ''),
                'nominal': float(target) if target else None,
                'low': band[0] if band else None,
                'high': band[1] if band else None,
                'value': float(value) if value else None,
                'status_from_file': text(measurement, './qif:Status/qif:CharacteristicStatusEnum'),
                'entities': tuple(sorted(set(entities))),
            })

    table = pd.DataFrame(rows)
    table['fraction'] = table.apply(fraction_of_band, axis=1)
    return table

The rows for this callout on this part: four holes, four faces. The first row is the
measurement followed by hand above:

In [14]:
table = qifkit.measurement_table(doc)
serial = qifkit.serial_number(doc, results, measurement)
table[(table.serial == serial) & (table.name == 'Radial Dimension (188)')]

,serial,characteristic,name,type,nominal,low,high,value,status_from_file,entities,fraction
12,SN#1,Diameter Item 5677,Radial Dimension (188),Diameter,8.73125,-0.254,0.254,8.800174,PASS,"(1116,)",0.271353
13,SN#1,Diameter Item 5679,Radial Dimension (188),Diameter,8.73125,-0.254,0.254,8.793088,PASS,"(1110,)",0.243455
14,SN#1,Diameter Item 5681,Radial Dimension (188),Diameter,8.73125,-0.254,0.254,8.778964,PASS,"(1122,)",0.187850
15,SN#1,Diameter Item 5683,Radial Dimension (188),Diameter,8.73125,-0.254,0.254,8.764469,PASS,"(1128,)",0.130784


## Where next

| Notebook | What it adds |
|---|---|
| [`01-tolerance-tightness`](01-tolerance-tightness.ipynb) | The callout join, drawn on the 3D part: design intent |
| [`02-inspection-report`](02-inspection-report.ipynb) | The measurement table, cleaned, as one part's inspection report |
| [`03-across-parts`](03-across-parts.ipynb) | All 15 parts: the pivot, bias beside scatter, the four holes |
| [`04-results-on-the-part`](04-results-on-the-part.ipynb) | The measurement join, drawn on the 3D part |
| [`05-probe-points-and-form`](05-probe-points-and-form.ipynb) | The probe points behind the numbers, and form nothing on the drawing controls |